In [ ]:

# !pip install pandas numpy matplotlib seaborn scikit-learn tensorflow


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# Cargar el dataset
url = "https://raw.githubusercontent.com/jdvelasq/datalake/main/ciencia_datos/datasets/precios_combustibles.csv"
df = pd.read_csv(url, sep=';', encoding='utf-8')
df['fecha_vigencia'] = pd.to_datetime(df['fecha_vigencia'], errors='coerce')
df = df.dropna(subset=['fecha_vigencia', 'precio'])

# Filtrar provincia
df = df[df['provincia'].str.upper() == 'BUENOS AIRES']

# Selección de columnas relevantes
df_modelo = df[['fecha_vigencia', 'precio', 'producto', 'tipohorario', 'empresabandera']].copy()

# Crear variables temporales
df_modelo['anio'] = df_modelo['fecha_vigencia'].dt.year
df_modelo['mes'] = df_modelo['fecha_vigencia'].dt.month
df_modelo = df_modelo.drop(columns=['fecha_vigencia'])

df_modelo = df_modelo.dropna()


In [ ]:

X = df_modelo.drop('precio', axis=1)
y = df_modelo['precio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical = ['producto', 'tipohorario', 'empresabandera']
numerical = ['anio', 'mes']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', StandardScaler(), numerical)
])

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Regresión Lineal - MAE:", mae_lr, "MSE:", mse_lr, "R2:", r2_lr)


In [ ]:

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_enc.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train_enc.toarray(), y_train, validation_split=0.2, epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

y_pred_nn = model.predict(X_test_enc.toarray()).flatten()

mae_nn = mean_absolute_error(y_test, y_pred_nn)
mse_nn = mean_squared_error(y_test, y_pred_nn)
r2_nn = r2_score(y_test, y_pred_nn)

print("Red Neuronal - MAE:", mae_nn, "MSE:", mse_nn, "R2:", r2_nn)


In [ ]:

import pandas as pd
df_resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Red Neuronal (TF)'],
    'MAE': [mae_lr, mae_nn],
    'MSE': [mse_lr, mse_nn],
    'R2': [r2_lr, r2_nn]
})
print(df_resultados)
